# Email-Enron Hypergraph: loading, validation and visualization
This notebook reads the native timestamped-simplex files, constructs both the temporal and unique static hypergraphs, reports reproducible statistics, and produces paper-ready visualizations.

**Model:** vertices are Enron employee email addresses; each email is a hyperedge containing its sender and all included Enron recipients.

In [ ]:
# Colab setup
!pip -q install hypernetx
import os, tarfile, time, zipfile
from collections import Counter
from itertools import combinations
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_SEED = 42

## 1. Upload and extract the archive
Run the next cell and select `email-Enron.tar.gz`.

In [ ]:
ARCHIVE = '/content/email-Enron.tar.gz'
if not os.path.exists(ARCHIVE):
    from google.colab import files
    uploaded = files.upload()
    archive_name = next((n for n in uploaded if n.endswith('.tar.gz')), None)
    if archive_name is None:
        raise ValueError('Please upload email-Enron.tar.gz')
    ARCHIVE = archive_name

EXTRACT_DIR = '/content/enron_data'
os.makedirs(EXTRACT_DIR, exist_ok=True)
with tarfile.open(ARCHIVE, 'r:gz') as tar:
    tar.extractall(EXTRACT_DIR, filter='data')
DATA_DIR = os.path.join(EXTRACT_DIR, 'email-Enron')
print('Extracted:', sorted(os.listdir(DATA_DIR)))

## 2. Parse timestamped simplices
`nverts[k]` gives the size of email/hyperedge `k`; the corresponding number of consecutive entries is then read from `simplices`.

In [ ]:
def read_int_lines(path):
    with open(path, encoding='utf-8') as f:
        return [int(x.strip()) for x in f if x.strip()]

nverts = read_int_lines(os.path.join(DATA_DIR, 'email-Enron-nverts.txt'))
flat_nodes = read_int_lines(os.path.join(DATA_DIR, 'email-Enron-simplices.txt'))
timestamps = read_int_lines(os.path.join(DATA_DIR, 'email-Enron-times.txt'))

labels = {}
with open(os.path.join(DATA_DIR, 'email-Enron-node-labels.txt'), encoding='utf-8') as f:
    for line in f:
        node, address = line.rstrip().split(maxsplit=1)
        labels[int(node)] = address

assert len(nverts) == len(timestamps), 'Hyperedge-count/timestamp mismatch'
assert sum(nverts) == len(flat_nodes), 'Flattened membership list mismatch'

raw_temporal_edges, temporal_edges, offset = [], [], 0
for size in nverts:
    raw_edge = tuple(flat_nodes[offset:offset + size])
    raw_temporal_edges.append(raw_edge)
    temporal_edges.append(tuple(sorted(raw_edge)))  # undirected hyperedge
    offset += size

record_unique_edges = set(raw_temporal_edges)
edge_frequency = Counter(temporal_edges)
unique_edges = list(edge_frequency)
vertices = sorted(set().union(*map(set, unique_edges)))

print(f'Employees (vertices): {len(vertices):,}')
print(f'Timestamped emails: {len(temporal_edges):,}')
print(f'Unique recorded simplices (order retained): {len(record_unique_edges):,}')
print(f'Unique undirected hyperedges (order removed): {len(unique_edges):,}')
print(f'Maximum hyperedge size: {max(map(len, unique_edges))}')

## 3. Incidence matrix and descriptive statistics

In [ ]:
vertex_index = {v: i for i, v in enumerate(vertices)}
rows, cols = [], []
for j, edge in enumerate(unique_edges):
    for v in edge:
        rows.append(vertex_index[v]); cols.append(j)
H = csr_matrix((np.ones(len(rows), dtype=np.uint8), (rows, cols)),
               shape=(len(vertices), len(unique_edges)))
hyperdegree = np.asarray(H.sum(axis=1)).ravel()
weighted_hyperdegree = np.array([sum(edge_frequency[e] for e in unique_edges if v in e)
                                 for v in vertices])
edge_sizes = np.asarray(H.sum(axis=0)).ravel()

summary = pd.DataFrame({
    'Statistic': ['Vertices', 'Timestamped hyperedges', 'Unique recorded simplices',
                  'Unique undirected hyperedges',
                  'Incidences', 'Mean unique-edge size', 'Median unique-edge size',
                  'Maximum edge size'],
    'Value': [len(vertices), len(temporal_edges), len(record_unique_edges),
              len(unique_edges), H.nnz,
              edge_sizes.mean(), np.median(edge_sizes), edge_sizes.max()]
})
display(summary)

node_table = pd.DataFrame({
    'node_id': vertices,
    'email': [labels[v] for v in vertices],
    'unique_hyperdegree': hyperdegree.astype(int),
    'email_participations': weighted_hyperdegree.astype(int)
}).sort_values('unique_hyperdegree', ascending=False)
display(node_table.head(15))

## 4. Hyperedge-size and hyperdegree distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(edge_sizes, discrete=True, ax=axes[0], color='#E76F51')
axes[0].set(title='Unique hyperedge-size distribution', xlabel='Employees in a hyperedge', ylabel='Number of hyperedges')
sns.histplot(hyperdegree, bins=25, ax=axes[1], color='#2A9D8F')
axes[1].set(title='Vertex hyperdegree distribution', xlabel='Number of unique incident hyperedges', ylabel='Number of employees')
plt.tight_layout()
plt.savefig('enron_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Readable hypergraph visualization
The complete hypergraph is too dense for a meaningful static figure. This incidence/bipartite view uses the 15 highest-hyperdegree employees and only hyperedges containing at least two of them. Squares are hyperedges, circles are employees, and square size reflects email frequency.

In [ ]:
TOP_N = 15
top_vertices = node_table.head(TOP_N)['node_id'].tolist()
selected_edges = [e for e in unique_edges if len(set(e) & set(top_vertices)) >= 2]
selected_edges = sorted(selected_edges, key=lambda e: edge_frequency[e], reverse=True)[:20]

B = nx.Graph()
for v in top_vertices:
    B.add_node(('v', v), bipartite=0)
for k, edge in enumerate(selected_edges, 1):
    h = ('e', k)
    B.add_node(h, bipartite=1, frequency=edge_frequency[edge])
    for v in set(edge) & set(top_vertices):
        B.add_edge(('v', v), h)

left = [('v', v) for v in top_vertices]
right = [('e', k) for k in range(1, len(selected_edges) + 1)]
pos = {}
for i, n in enumerate(left): pos[n] = (0, -i)
for i, n in enumerate(right): pos[n] = (1, -i * max(1, (len(left)-1)/(max(1, len(right)-1))))

plt.figure(figsize=(15, 10))
nx.draw_networkx_edges(B, pos, alpha=.25, width=.8, edge_color='#6C757D')
nx.draw_networkx_nodes(B, pos, nodelist=left, node_color='#2A9D8F', node_size=500, edgecolors='white')
right_frequency = [B.nodes[n]['frequency'] for n in right]
nx.draw_networkx_nodes(B, pos, nodelist=right, node_color=right_frequency, cmap='OrRd', node_shape='s', node_size=420, edgecolors='white')
left_labels = {('v', v): labels[v].split('@')[0] for v in top_vertices}
right_labels = {('e', k): f'e{k}  (×{edge_frequency[selected_edges[k-1]]})' for k in range(1, len(selected_edges)+1)}
nx.draw_networkx_labels(B, {n:(x-.025,y) for n,(x,y) in pos.items() if n in left}, left_labels, horizontalalignment='right', font_size=8)
nx.draw_networkx_labels(B, {n:(x+.025,y) for n,(x,y) in pos.items() if n in right}, right_labels, horizontalalignment='left', font_size=7, font_color='#7F2704', bbox=dict(facecolor='white', edgecolor='none', alpha=.8, pad=.2))
plt.title('Email-Enron filtered incidence view: employees and group emails')
plt.axis('off'); plt.tight_layout()
plt.savefig('enron_filtered_hypergraph.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Weighted two-section projection for structural visualization
An edge connects two employees when they occur in at least one common hyperedge. Its weight is the number of unique hyperedges they share. This projection is used only for visualization and pairwise distance-based calculations; the native incidence structure remains the primary data representation.

In [ ]:
G = nx.Graph()
G.add_nodes_from(vertices)
for edge in unique_edges:
    for u, v in combinations(edge, 2):
        if G.has_edge(u, v): G[u][v]['weight'] += 1
        else: G.add_edge(u, v, weight=1)

top_set = set(top_vertices)
SG = G.subgraph(top_set).copy()
pos = nx.spring_layout(SG, seed=RANDOM_SEED, weight='weight')
degrees = dict(SG.degree(weight='weight'))
widths = [0.5 + 2.5*np.log1p(SG[u][v]['weight'])/np.log1p(max(nx.get_edge_attributes(SG,'weight').values())) for u,v in SG.edges()]
plt.figure(figsize=(11, 8))
nx.draw_networkx_edges(SG, pos, width=widths, alpha=.3, edge_color='#457B9D')
nx.draw_networkx_nodes(SG, pos, node_size=[250+10*degrees[v] for v in SG], node_color=[hyperdegree[vertex_index[v]] for v in SG], cmap='viridis', edgecolors='white')
for v, (x, y) in pos.items():
    plt.annotate(labels[v].split('@')[0], (x, y), xytext=(0, 9), textcoords='offset points', ha='center', va='bottom', fontsize=7, bbox=dict(boxstyle='round,pad=.18', facecolor='white', edgecolor='none', alpha=.82))
plt.title('Top-15 employees: weighted two-section projection')
plt.axis('off'); plt.tight_layout()
plt.savefig('enron_top15_projection.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Projected graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 7. Baseline centralities and computational time
Hypergraph distance is represented by shortest-path distance in the unweighted two-section: two vertices have distance 1 when they share a hyperedge.

In [ ]:
timings, scores = {}, {}

t = time.perf_counter(); scores['Hyperdegree'] = dict(zip(vertices, hyperdegree)); timings['Hyperdegree'] = time.perf_counter()-t
t = time.perf_counter(); scores['Closeness'] = nx.closeness_centrality(G); timings['Closeness'] = time.perf_counter()-t
t = time.perf_counter(); scores['Betweenness'] = nx.betweenness_centrality(G, normalized=True); timings['Betweenness'] = time.perf_counter()-t
t = time.perf_counter(); scores['Eigenvector'] = nx.eigenvector_centrality_numpy(G, weight='weight'); timings['Eigenvector'] = time.perf_counter()-t

results = pd.DataFrame({'node_id': vertices, 'email': [labels[v] for v in vertices]})
for name, values in scores.items(): results[name] = results.node_id.map(values)
display(results.sort_values('Hyperdegree', ascending=False).head(15))
display(pd.DataFrame({'Measure': timings.keys(), 'Seconds': timings.values()}))

## 8. Hypergraph Dangling Centrality
The communication strength is $\Phi(H)=\sum_{i<j}1/d_H(v_i,v_j)$, with unreachable pairs contributing zero. For vertex $v$, dangling centrality is $[\Phi(H)-\Phi(H^{-v})]/\Phi(H)$. Here, removing a vertex deletes it from every incident hyperedge while retaining it as an isolated vertex.

In [ ]:
def communication_strength(graph):
    total = 0.0
    for source, dist in nx.all_pairs_shortest_path_length(graph):
        total += sum(1/d for target, d in dist.items() if source < target and d > 0)
    return total

phi_original = communication_strength(G)
dangling = {}
t0 = time.perf_counter()
for idx, removed in enumerate(vertices, 1):
    modified_edges = [tuple(v for v in edge if v != removed) for edge in unique_edges]
    G_minus = nx.Graph(); G_minus.add_nodes_from(vertices)
    for edge in modified_edges:
        for u, v in combinations(edge, 2): G_minus.add_edge(u, v)
    phi_minus = communication_strength(G_minus)
    dangling[removed] = (phi_original - phi_minus) / phi_original
    if idx % 20 == 0: print(f'Processed {idx}/{len(vertices)} vertices')
dangling_time = time.perf_counter() - t0
results['Dangling'] = results.node_id.map(dangling)
print(f'Dangling-centrality runtime: {dangling_time:.3f} seconds')
display(results.sort_values('Dangling', ascending=False).head(15))

## 9. Comparison heatmap and export

In [ ]:
measure_cols = ['Hyperdegree','Closeness','Betweenness','Eigenvector','Dangling']
corr = results[measure_cols].corr(method='spearman')
plt.figure(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-1, vmax=1)
plt.title('Spearman rank correlations among centrality measures')
plt.tight_layout(); plt.savefig('enron_centrality_correlations.png', dpi=300, bbox_inches='tight'); plt.show()

results.to_csv('email_enron_hypergraph_centralities.csv', index=False)
pd.DataFrame({'Measure': list(timings)+['Dangling'], 'Seconds': list(timings.values())+[dangling_time]}).to_csv('email_enron_runtime.csv', index=False)
output_files = [
    'email_enron_hypergraph_centralities.csv', 'email_enron_runtime.csv',
    'enron_distributions.png', 'enron_filtered_hypergraph.png',
    'enron_top15_projection.png', 'enron_centrality_correlations.png'
]
with zipfile.ZipFile('Email_Enron_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for filename in output_files: zf.write(filename)
print('Saved two CSV files, four 300-dpi figures, and Email_Enron_results.zip')

# In Google Colab, remove # from the following two lines to download everything:
# from google.colab import files
# files.download('Email_Enron_results.zip')